In [1]:
import yfinance as yf
import pandas as pd
import numpy as np

## Data Preparation

In [2]:
# pip install curl_cffi

In [3]:
#pip install -U yfinance

In [4]:
from curl_cffi import requests as cf_requests
import yfinance as yf

session = cf_requests.Session(impersonate="chrome")


tickers = ["AIR.PA", "SAF.PA"]

prices = yf.download(
    tickers,
    start="2021-01-01",
    auto_adjust=True,
    progress=False,
    session=session
)["Close"]

prices.head()

Ticker,AIR.PA,SAF.PA
Date,,
2021-01-04,82.468246,110.404167
2021-01-05,82.413200,110.641785
2021-01-06,82.725128,110.546738
2021-01-07,82.220535,110.071480
2021-01-08,82.660904,110.594261


In [5]:
#prices = prices.rename(columns={"AIR.PA": "AIR", "SAF.PA": "SAF"})

In [6]:
print(prices.columns, prices.size, prices.shape)  

Index(['AIR.PA', 'SAF.PA'], dtype='object', name='Ticker') 2866 (1433, 2)


In [7]:
prices.to_csv("prices.csv")

# later, to load it back:
prices = pd.read_csv("prices.csv", index_col="Date", parse_dates=True)

## Data Loading / Analysis

In [8]:
prices = pd.read_csv("prices.csv", index_col="Date", parse_dates=True)
#prices.head()

# Create ratio DataFrame
ratio_df = pd.DataFrame(index=prices.index)
col1, col2 = prices.columns

ratio_df["Ratio"] = prices[col1] / prices[col2]

ratio_df.tail()

ratio_df["Ratio_Return"] = ratio_df["Ratio"].pct_change()

ratio_df.dropna(inplace=True)
ratio_df.tail()

,Ratio,Ratio_Return
Date,,
2026-07-31,0.597518,-0.003258
2026-08-03,0.592087,-0.009090
2026-08-04,0.600730,0.014598
2026-08-05,0.599582,-0.001911
2026-08-06,0.598114,-0.002448


In [9]:
levels = [1.3,1.2,1.05,0.9,0.8,0.75,0.7,0.65,0.6,0.55,0.5]
sorted_levels = np.sort(np.array(levels))


print("list:",levels,"\nsorted array:",sorted_levels)


list: [1.3, 1.2, 1.05, 0.9, 0.8, 0.75, 0.7, 0.65, 0.6, 0.55, 0.5] 
sorted array: [0.5  0.55 0.6  0.65 0.7  0.75 0.8  0.9  1.05 1.2  1.3 ]


# **REVERSION SIGNAL GENERATION**

## **1. FUNCTION TO FIND THE SUPPORT AND RESISTANCE FOR EACH ROW**

In [10]:
def find_support_resistance(ratio_series, levels):

    supports = []
    resistances = []

    for price in ratio_series:
        below = levels[levels < price]
        above = levels[levels > price]

        support = below.max() if len(below) > 0 else np.nan
        resistance = above.min() if len(above) > 0 else np.nan

        supports.append(support)
        resistances.append(resistance)

    result = pd.DataFrame({
        "Ratio": ratio_series,
        "Support": supports,
        "Resistance": resistances
    })

    return result

In [11]:
df = find_support_resistance(ratio_df["Ratio"], sorted_levels)

## **2. FUNCTION TO GET THE NORMALIZED PX**

In [12]:
def add_normalized_price(df):
    df = df.copy()
    df["norm_price"] = (df["Ratio"] - df["Support"]) / (df["Resistance"] - df["Support"])
    return df

In [13]:
df = add_normalized_price(df)

## **3. FUNCTION TO GET THE BOUNCE ZONE**

In [14]:
def get_zone(norm):
    if 0.10 <= norm <= 0.35:
        return "rev sup bounce"
    elif 0.65 <= norm <= 0.90:
        return "rev res bounce"
    else:
        return None

In [15]:
def add_bounce_zone(df):
    df = df.copy()
    df["bounce_zone"] = df["norm_price"].apply(get_zone)
    return df

In [16]:
df = add_bounce_zone(df)

## **4. FUNCTION FOR TOUCH CHECK**

In [17]:
def add_touch_check(df):
    df = df.copy()
    
    touch_days_ago = []

    for i in range(len(df)):
        zone = df["bounce_zone"].iloc[i]
        support = df["Support"].iloc[i]
        resistance = df["Resistance"].iloc[i]

        touch = None # default value

        if zone == "rev res bounce":
            for days_back in range(5,0,-1):
                price = df["Ratio"].iloc[i - days_back]
                if abs(price - resistance) / resistance <= 0.005:
                    touch = days_back
                    break

        elif zone == "rev sup bounce":
            for days_back in [5, 4, 3, 2, 1]:
                price = df["Ratio"].iloc[i - days_back]
                if abs(price - support) / support <= 0.005:
                    touch = days_back
                    break

        touch_days_ago.append(touch)

    df["touch_days_ago"] = touch_days_ago
    return df

In [18]:
df = add_touch_check(df)

## **5. FUNCTION FOR 15D AVG PX BEFORE TOUCH**

In [19]:
def add_avg_15d_at_touch(df):
    df = df.copy()
    
    avg_15d = []
    
    for i in range(len(df)):
        touch = df["touch_days_ago"].iloc[i]
        
        if pd.isna(touch):
            avg_15d.append(None)
            continue
        
        touch = int(touch)  # in case it got stored as float (e.g. 5.0)
        touch_index = i - touch
        
        if touch_index - 15 < 0:
            avg_15d.append(None)
            continue
        
        window = df["Ratio"].iloc[touch_index - 15 : touch_index]
        avg_15d.append(window.mean())
    
    df["avg_15d_at_touch"] = avg_15d
    return df

In [20]:
df = add_avg_15d_at_touch(df)

## **6. FUNCTION TO NORMALIZE 15D AVG PX BEFORE TOUCH**

In [21]:
def add_norm_avg(df):
    df = df.copy()
    df["norm_avg"] = (df["avg_15d_at_touch"] - df["Support"]) / (df["Resistance"] - df["Support"])
    return df

In [22]:
df = add_norm_avg(df)

## **7. FUNCTION FOR APPROACH CHECK**

In [23]:
def add_approach_check(df):
    df = df.copy()
    
    approach = []
    
    for i in range(len(df)):
        zone = df["bounce_zone"].iloc[i]
        norm_avg = df["norm_avg"].iloc[i]
        
        if pd.isna(norm_avg):
            approach.append(None)
        elif zone == "rev res bounce" and 0 < norm_avg < 0.8:
            approach.append("uptrend approach")
        elif zone == "rev sup bounce" and 0.2 < norm_avg < 1:
            approach.append("downtrend approach")
        else:
            approach.append(None)
    
    df["approach"] = approach
    return df

In [24]:
df = add_approach_check(df)

## **8. FUNCTION FOR SIGNAL V1**

In [25]:
def add_signal_v1(df):
    df = df.copy()
    
    signal = []
    
    for i in range(len(df)):
        zone = df["bounce_zone"].iloc[i]
        approach = df["approach"].iloc[i]
        touch = df["touch_days_ago"].iloc[i]
        
        has_touch = not pd.isna(touch)
        
        if zone == "rev res bounce" and approach == "uptrend approach" and has_touch:
            signal.append("reverting from resistance")
        elif zone == "rev sup bounce" and approach == "downtrend approach" and has_touch:
            signal.append("reverting from support")
        else:
            signal.append(None)
    
    df["signal_v1"] = signal
    return df

In [26]:
df = add_signal_v1(df)

## **9. FUNCTION FOR BLOCK1: NO PRIOR BREACH IN TOUCH ZONE (5D)**

In [27]:
def add_block1_check(df):
    df = df.copy()
    
    block1_pass = []
    
    for i in range(len(df)):
        signal = df["signal_v1"].iloc[i]
        resistance = df["Resistance"].iloc[i]
        support = df["Support"].iloc[i]
        
        if signal is None:
            block1_pass.append(None)
            continue
        
        if i < 5:
            block1_pass.append(None)
            continue
        
        window = df["Ratio"].iloc[i-5:i]  # last 5 days, excluding live day
        
        if signal == "reverting from resistance":
            passed = all(window < 1.03 * resistance)                # if even one day breaks it, all() returns False
        elif signal == "reverting from support":
            passed = all(window > 0.97 * support)                   # if even one day breaks it, all() returns False
        else:
            passed = None
        
        block1_pass.append(passed)
    
    df["block1_pass"] = block1_pass
    return df

In [28]:
df = add_block1_check(df)

## **10. FUNCTION FOR BLOCK2: YESTERDAY IN ZONE CHECK**

In [29]:
def add_block2_check(df):
    df = df.copy()
    
    block2_pass = []
    
    for i in range(len(df)):
        signal = df["signal_v1"].iloc[i]
        resistance = df["Resistance"].iloc[i]
        support = df["Support"].iloc[i]
        
        if signal is None:
            block2_pass.append(None)
            continue
        
        if i < 1:
            block2_pass.append(None)
            continue
        
        yesterday_price = df["Ratio"].iloc[i-1]
        y_norm = (yesterday_price - support) / (resistance - support)
        
        if signal == "reverting from resistance":
            passed = 0.65 <= y_norm <= 0.90
        elif signal == "reverting from support":
            passed = 0.10 <= y_norm <= 0.35
        else:
            passed = None
        
        block2_pass.append(passed)
    
    df["block2_pass"] = block2_pass
    return df

In [30]:
df = add_block2_check(df)

## **FINAL SIGNAL**

In [31]:
def add_final_signal(df):
    df = df.copy()
    
    final_signal = []
    
    for i in range(len(df)):
        signal = df["signal_v1"].iloc[i]
        b1 = df["block1_pass"].iloc[i]
        b2 = df["block2_pass"].iloc[i]
        
        if signal is not None and b1 == True and b2 == True:
            final_signal.append(signal)
        else:
            final_signal.append(None)
    
    df["final_signal"] = final_signal
    return df

In [32]:
df = add_final_signal(df)

In [33]:
df[~df["final_signal"].isna()].tail()

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch,norm_avg,approach,signal_v1,block1_pass,block2_pass,final_signal
Date,,,,,,,,,,,,,
2025-12-08,0.666683,0.65,0.7,0.333666,rev sup bounce,4.0,0.685531,0.710621,downtrend approach,reverting from support,True,True,reverting from support
2025-12-09,0.658244,0.65,0.7,0.164881,rev sup bounce,5.0,0.685531,0.710621,downtrend approach,reverting from support,True,True,reverting from support
2026-07-16,0.594409,0.55,0.6,0.888179,rev res bounce,2.0,0.578884,0.577670,uptrend approach,reverting from resistance,True,True,reverting from resistance
2026-07-17,0.589742,0.55,0.6,0.794841,rev res bounce,3.0,0.578884,0.577670,uptrend approach,reverting from resistance,True,True,reverting from resistance
2026-07-20,0.590637,0.55,0.6,0.812750,rev res bounce,4.0,0.578884,0.577670,uptrend approach,reverting from resistance,True,True,reverting from resistance


In [34]:
# Save
df.to_csv("reversion_signals.csv", index=True)

# Read back later
df = pd.read_csv("reversion_signals.csv", index_col=0, parse_dates=True)

# **PROBABILITY ANALYSIS**

In [35]:
df = pd.read_csv("reversion_signals.csv", index_col=0, parse_dates=True)

In [36]:
df.tail()

,Ratio,Support,Resistance,norm_price,bounce_zone,touch_days_ago,avg_15d_at_touch,norm_avg,approach,signal_v1,block1_pass,block2_pass,final_signal
Date,,,,,,,,,,,,,
2026-07-31,0.597518,0.55,0.60,0.950355,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-03,0.592087,0.55,0.60,0.841731,rev res bounce,2.0,0.605502,1.110032,NaN,NaN,NaN,NaN,NaN
2026-08-04,0.600730,0.60,0.65,0.014591,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-05,0.599582,0.55,0.60,0.991632,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2026-08-06,0.598114,0.55,0.60,0.962274,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [37]:
probability_df = df[["Ratio","Support","Resistance","final_signal"]]
probability_df.tail()

,Ratio,Support,Resistance,final_signal
Date,,,,
2026-07-31,0.597518,0.55,0.60,NaN
2026-08-03,0.592087,0.55,0.60,NaN
2026-08-04,0.600730,0.60,0.65,NaN
2026-08-05,0.599582,0.55,0.60,NaN
2026-08-06,0.598114,0.55,0.60,NaN


In [38]:
probability_df[probability_df["final_signal"] == "reverting from support"].shape

(21, 4)

In [39]:
probability_df[probability_df["final_signal"] == "reverting from resistance"].shape

(15, 4)

In [40]:
results = []

for i in range(len(probability_df)):

    row = probability_df.iloc[i]
    signal = row["final_signal"]

    if signal not in ["reverting from support","reverting from resistance"]:
        continue

    # store entry info
    entry_date = probability_df.index[i]
    entry_price = row["Ratio"]

    if signal == "reverting from support":
        target = row["Resistance"]
        stop = row["Support"]

    if signal == "reverting from resistance":
            target = row["Support"]
            stop = row["Resistance"]

    for j in range(i+1, len(probability_df)):
         

        future_price = probability_df.iloc[j]["Ratio"]
        exit_date = probability_df.index[j]

        outcome = None

        # Support trade
        if signal == "reverting from support":

            if future_price >= target:
                outcome = "Target"
            elif future_price <= stop:
                outcome = "Stop"

        # Resistance trade
        if signal == "reverting from resistance":

            if future_price <= target:
                outcome = "Target"
            elif future_price >= stop:
                outcome = "Stop"


        if outcome is not None:
            results.append({
                "Entry Date": entry_date,
                "Entry Px": entry_price,
                "Signal": signal,
                "Exit Date": exit_date,
                "Exit Px": future_price,
                "Outcome": outcome,
                "Holding Days": j-i
            })

            break



In [41]:
results_df = pd.DataFrame(results)

In [42]:
results_df

,Entry Date,Entry Px,Signal,Exit Date,Exit Px,Outcome,Holding Days
0,2022-12-02,0.919541,reverting from support,2023-01-11,0.898196,Stop,27
1,2022-12-05,0.918462,reverting from support,2023-01-11,0.898196,Stop,26
2,2022-12-06,0.926766,reverting from support,2023-01-11,0.898196,Stop,25
3,2023-02-21,0.887370,reverting from resistance,2023-06-02,0.900263,Stop,70
4,2023-02-22,0.888536,reverting from resistance,2023-06-02,0.900263,Stop,69
5,2023-02-27,0.889719,reverting from resistance,2023-06-02,0.900263,Stop,66
6,2023-03-07,0.886669,reverting from resistance,2023-06-02,0.900263,Stop,60
7,2023-09-11,0.868341,reverting from resistance,2023-10-13,0.797645,Target,24
8,2023-10-18,0.812861,reverting from support,2024-02-15,0.792639,Stop,83
9,2023-10-19,0.813717,reverting from support,2024-02-15,0.792639,Stop,82


Removing repeating signals (consecutive)

In [43]:
results_df["Entry Date"] = pd.to_datetime(results_df["Entry Date"])

results_df["prev_date"] = results_df["Entry Date"].shift(1)

results_df["date_gap"] = (results_df["Entry Date"] - results_df["prev_date"]).dt.days

results_df["is_new_trade"] = results_df["date_gap"].isna() | (results_df["date_gap"] > 3)

results_df_deduped = results_df[results_df["is_new_trade"]]

results_df_deduped

,Entry Date,Entry Px,Signal,Exit Date,Exit Px,Outcome,Holding Days,prev_date,date_gap,is_new_trade
0,2022-12-02,0.919541,reverting from support,2023-01-11,0.898196,Stop,27,NaT,NaN,True
3,2023-02-21,0.887370,reverting from resistance,2023-06-02,0.900263,Stop,70,2022-12-06,77.0,True
5,2023-02-27,0.889719,reverting from resistance,2023-06-02,0.900263,Stop,66,2023-02-22,5.0,True
6,2023-03-07,0.886669,reverting from resistance,2023-06-02,0.900263,Stop,60,2023-02-27,8.0,True
7,2023-09-11,0.868341,reverting from resistance,2023-10-13,0.797645,Target,24,2023-03-07,188.0,True
8,2023-10-18,0.812861,reverting from support,2024-02-15,0.792639,Stop,83,2023-09-11,37.0,True
12,2023-11-22,0.812008,reverting from support,2024-02-15,0.792639,Stop,58,2023-10-23,30.0,True
15,2024-04-02,0.793880,reverting from resistance,2024-04-26,0.746896,Target,18,2023-11-24,130.0,True
18,2024-05-09,0.759018,reverting from support,2024-05-10,0.746166,Stop,1,2024-04-04,35.0,True
19,2024-10-08,0.606985,reverting from support,2024-10-22,0.659219,Target,10,2024-05-09,152.0,True


In [44]:
df = results_df_deduped.copy()

# 1. sign column: +1 for long (support), -1 for short (resistance)
df["sign"] = df["Signal"].map({"reverting from support": 1, "reverting from resistance": -1})

# 2. returns column
df["return_pct"] = df["sign"] * (df["Exit Px"] - df["Entry Px"]) / df["Entry Px"]

df.tail()

,Entry Date,Entry Px,Signal,Exit Date,Exit Px,Outcome,Holding Days,prev_date,date_gap,is_new_trade,sign,return_pct
23,2025-05-08,0.635602,reverting from resistance,2025-06-26,0.654126,Stop,35,2025-03-07,62.0,True,-1,-0.029144
26,2025-08-06,0.608876,reverting from support,2025-09-03,0.651231,Target,20,2025-05-12,86.0,True,1,0.069562
27,2025-10-02,0.660782,reverting from support,2025-11-24,0.703106,Target,37,2025-08-06,57.0,True,1,0.064052
29,2025-12-04,0.665371,reverting from support,2025-12-17,0.643671,Stop,9,2025-10-03,62.0,True,1,-0.032614
33,2026-07-16,0.594409,reverting from resistance,2026-07-21,0.602663,Stop,3,2025-12-09,219.0,True,-1,-0.013887


## Probability and Statistics

In [45]:
df[df["Outcome"] == "Target"].shape[0] / df.shape[0]

winrate = df[df["Outcome"] == "Target"].shape[0] / df.shape[0]


In [46]:
df.loc[df['Outcome'] == "Target", "return_pct"]
df.loc[df['Outcome'] == "Target", "return_pct"].mean()

avgwin = df.loc[df['Outcome'] == "Target", "return_pct"].mean()

#print(f"Avg win%: {avgwin*100:.2f}%")

In [47]:
df.loc[df['Outcome'] == "Stop", "return_pct"]
df.loc[df['Outcome'] == "Stop", "return_pct"].mean()

avgloss = df.loc[df['Outcome'] == "Stop", "return_pct"].mean()

#print(f"Avg loss%: {avgloss*100:.2f}%")

In [48]:
ev = winrate*avgwin + (1-winrate)*avgloss
#print(f"Expected value per trade: {ev*100:.2f}%")

In [49]:
# OUTPUT
print(f"Output")
print(f"======\n")

print(f"Total unique signals: {df.shape[0]}")
print(f"Win rate: {winrate*100:.0f}%")
print(f"Avg win%: {avgwin*100:.2f}%")
print(f"Avg loss%: {avgloss*100:.2f}%")
print(f"Expected value per trade: {ev*100:.2f}%")

Output

Total unique signals: 17
Win rate: 29%
Avg win%: 7.21%
Avg loss%: -2.25%
Expected value per trade: 0.53%


## Grouped by Signal

In [50]:
wins = df[df["Outcome"] == "Target"]
losses = df[df["Outcome"] == "Stop"]

win_rate = df.groupby("Signal")["Outcome"].apply(lambda x: (x == "Target").mean())
avg_gain = wins.groupby("Signal")["return_pct"].mean()
avg_loss = losses.groupby("Signal")["return_pct"].mean()

ev = (win_rate * avg_gain + (1 - win_rate) * avg_loss)*100


summary = pd.DataFrame({
    "win_rate%": (win_rate*100).round(0),
    "avg_gain%": (avg_gain*100).round(1),
    "avg_loss%": (avg_loss*100).round(1),
    "ev%": ev.round(2)
})

summary

,win_rate%,avg_gain%,avg_loss%,ev%
Signal,,,,
reverting from resistance,25.0,7.0,-2.0,0.29
reverting from support,33.0,7.3,-2.5,0.75


## Average holding period for hitting target

In [51]:
df[df["Outcome"] == "Target"]["Holding Days"].mean()

avg_hld = df[df["Outcome"] == "Target"]["Holding Days"].mean()
median_hld = df[df["Outcome"] == "Target"]["Holding Days"].median()

print(f"Average holding period to reach target: {avg_hld} days")
print(f"Median holding period to reach target: {median_hld} days")

Average holding period to reach target: 21.8 days
Median holding period to reach target: 20.0 days


In [52]:
df[df["Outcome"] == "Stop"]["Holding Days"].mean()

avg_stop = df[df["Outcome"] == "Stop"]["Holding Days"].mean()
median_stop = df[df["Outcome"] == "Stop"]["Holding Days"].median()

print(f"Average holding period to hit stop: {round(avg_stop,1)} days")
print(f"Median holding period to hit stop: {round(median_stop,1)} days")

Average holding period to hit stop: 34.7 days
Median holding period to hit stop: 31.0 days


In [53]:
df.groupby("Signal")["Holding Days"].mean()
df.groupby("Signal")["Holding Days"].median()

Signal
reverting from resistance    29.5
reverting from support       20.0
Name: Holding Days, dtype: float64

In [54]:
summary = df.groupby(["Signal", "Outcome"])["Holding Days"].agg(["mean", "median", "count"])
summary["mean"] = summary["mean"].round(1)
summary

mean  median  count
Signal                    Outcome                     
reverting from resistance Stop     39.2    47.5      6
                          Target   21.0    21.0      2
reverting from support    Stop     30.2    18.0      6
                          Target   22.3    20.0      3